# Datenquelle Chicago: Crimes – 2001 to Present

**Zweck:** Dieses Notebook dokumentiert die Chicago-Quelle, bevor wir Daten laden oder transformieren. Es beschreibt, woher die Daten stammen, wie sie technisch bereitgestellt werden und was eine Zeile fachlich bedeutet.

**Stand der Dokumentation:** 3. September 2026

## 1. Kurzüberblick

| Merkmal | Beschreibung |
|---|---|
| Datensatz | **Crimes – 2001 to Present** |
| Herausgeber | City of Chicago Data Portal |
| Datenverantwortlicher | Chicago Police Department (CPD) |
| Quellsystem | CLEAR – Citizen Law Enforcement Analysis and Reporting |
| Plattform | Socrata Open Data |
| Schnittstelle | SODA – Socrata Open Data API |
| Dataset-ID | `ijzp-q8t2` |
| Zeitraum | 2001 bis heute, ohne die jeweils jüngsten sieben Tage |
| Aktualisierung | täglich |
| Kategorie im Portal | Public Safety |
| Räumlicher Bezug | City of Chicago |

## 2. Um welche Art von Quelle handelt es sich?

Wir erhalten **keinen direkten Zugriff auf die operative Polizeidatenbank CLEAR**. Das Chicago Police Department extrahiert Daten aus CLEAR und veröffentlicht sie im Open-Data-Portal der Stadt.

Dort liegt die Quelle als **tabellarischer Socrata-Datensatz** vor: fachlich ähnlich einer grossen Tabelle mit Zeilen und Spalten. Socrata stellt diese Tabelle zusätzlich über eine HTTP-API bereit. Die API heisst **SODA (Socrata Open Data API)** und kann Daten unter anderem als JSON, CSV oder GeoJSON liefern.

Der Datenweg ist damit:

`CPD CLEAR → Extraktion durch CPD → Chicago Data Portal / Socrata → SODA API → unser Python-Projekt`

Die SODA API unterstützt mit **SoQL** eine SQL-ähnliche Abfragesprache. Dadurch können wir bereits auf der Quellseite Spalten auswählen, Zeiträume und Deliktarten filtern, sortieren und aggregieren. Das reduziert unnötige Downloads.

## 3. Datensatz und fachliche Bedeutung einer Zeile

Der Datensatz beschreibt laut Chicago gemeldete Straftaten in der Stadt. Für die meisten Deliktarten steht eine Zeile grundsätzlich für einen **gemeldeten Crime Incident**. Die Spalte `id` identifiziert den Datensatz, während `case_number` die CPD Records Division Number des Vorfalls enthält.

### Sonderfall Murder/Homicide

Murder/Homicide ist nicht separat im Sinn von „im Crime-Dataset gar nicht vorhanden“. Chicago weist aber auf eine andere Granularität hin:

- Bei Murder/Homicide existiert ein Datensatz **pro Opfer**.
- Die übrigen Crimes werden grundsätzlich als gemeldete **Crime Incidents** geführt.

Damit besitzt `COUNT(*)` bei Homicide nicht dieselbe fachliche Bedeutung wie beispielsweise bei Robbery oder Burglary. Technisch sieht beides nach einer Zeile aus, fachlich repräsentiert die Zeile jedoch eine andere Einheit.

Für spätere Erweiterungen sollte diese Semantik explizit modelliert werden, zum Beispiel mit:

- `record_granularity = INCIDENT`
- `record_granularity = VICTIM`

## 4. Kategorien im ersten PoC (LN1)

Für Chicago werden die Deliktkategorien **nicht eingegrenzt**. Wir übernehmen grundsätzlich alle in `primary_type` vorkommenden Werte aus der Quelle. Die Kategorien werden nicht als feste Liste im Code hinterlegt, weil sich Quellwerte und Klassifikationen im Lauf der Zeit ändern können.

Dabei gelten folgende Regeln:

- Der ursprüngliche Wert aus `primary_type` bleibt unverändert erhalten.
- Eine spätere Vereinheitlichung für den Städtevergleich erfolgt in einer zusätzlichen Spalte; der Quellwert wird nicht überschrieben.
- Neue oder unbekannte Kategorien werden nicht verworfen.
- Homicide bleibt im Datensatz enthalten, erhält wegen der abweichenden Bedeutung einer Zeile aber die Kennzeichnung `record_granularity = VICTIM`.
- Die übrigen Kategorien erhalten grundsätzlich `record_granularity = INCIDENT`.

Homicide darf deshalb nicht ohne weitere Behandlung gemeinsam mit incident-basierten Kategorien über `COUNT(*)` verglichen oder summiert werden.

## 5. Wichtige Quellspalten

| Spalte | Fachliche Bedeutung | Voraussichtliche Nutzung |
|---|---|---|
| `id` | eindeutige Kennung des Datensatzes | technische Identifikation / Deduplizierung |
| `case_number` | CPD Records Division Number des Incidents | fachliche Referenz; Granularität prüfen |
| `date` | Zeitpunkt des Ereignisses; teilweise geschätzt | zeitliche Analyse |
| `block` | aus Datenschutzgründen teilweise anonymisierte Adresse | grobe räumliche Einordnung |
| `iucr` | Illinois Uniform Crime Reporting Code | detaillierte Deliktklassifikation |
| `primary_type` | primäre Beschreibung des IUCR-Codes | Übernahme aller vorhandenen Deliktkategorien |
| `description` | detailliertere Unterkategorie | optionale Detailanalyse |
| `location_description` | Art des Tatorts | optionale Kontextanalyse |
| `arrest` | ob eine Verhaftung erfolgte | optionales Analysemerkmal |
| `domestic` | Kennzeichnung eines häuslichen Bezugs | optionales Analysemerkmal |
| `district`, `ward`, `community_area` | administrative Gebiete | räumliche Aggregation |
| `year` | Jahr des Ereignisses | Filter und Partitionierung |
| `updated_on` | letzte Aktualisierung des Datensatzes | inkrementelle Verarbeitung / Nachvollziehbarkeit |
| `latitude`, `longitude`, `location` | approximative Geokoordinaten | Karten und räumliche Zuordnung |

Die vollständige Spaltenliste und die jeweils aktuellen Datentypen liefert der offizielle Metadaten-Endpunkt. Das spätere Ingestion-Skript soll sich nur auf die tatsächlich benötigten Spalten beschränken.

## 6. Technischer API-Zugriff

### Identifikation

- Portal-Domain: `data.cityofchicago.org`
- Dataset-ID: `ijzp-q8t2`
- Metadaten: `https://data.cityofchicago.org/api/views/ijzp-q8t2`

### SODA-Endpunkte

Socrata dokumentiert derzeit einen SODA-3-Abfrage-Endpunkt:

`https://data.cityofchicago.org/api/v3/views/ijzp-q8t2/query.json`

Daneben gibt es den klassischen ressourcenorientierten SODA-Endpunkt, der sich besonders einfach mit HTTP GET und SoQL-Parametern verwenden lässt:

`https://data.cityofchicago.org/resource/ijzp-q8t2.json`

Vor der Implementierung entscheiden wir uns bewusst für **einen** Endpunkt und halten den Zugriff einfach. In beiden Fällen werden nur benötigte Spalten und Zeiträume abgefragt; die Deliktkategorien werden nicht eingeschränkt. Grosse Resultate müssen seitenweise geladen werden; eine stabile Sortierung ist für reproduzierbares Paging wichtig.

### App-Token

Der Socrata App Token wird lokal als `SOCRATA_APP_TOKEN` in der Datei `.env` gespeichert. Er gehört weder in das Notebook noch in Git. Bei SODA wird der Token bevorzugt im HTTP-Header `X-App-Token` übertragen. Ein App Token ordnet Anfragen einer Anwendung zu und verbessert die verfügbaren Request-Limits; er ist nicht automatisch eine Benutzeranmeldung.

In [ ]:
# Zentrale, nicht geheime Kennwerte der Quelle
CHICAGO_SOURCE = {
    "domain": "data.cityofchicago.org",
    "dataset_id": "ijzp-q8t2",
    "dataset_name": "Crimes - 2001 to Present",
    "publisher": "City of Chicago",
    "attribution": "Chicago Police Department",
    "source_system": "CLEAR",
    "api": "SODA",
}

CHICAGO_SOURCE

## 7. Datenqualität, Datenschutz und Grenzen

Die folgenden Einschränkungen stammen aus der offiziellen Datensatzbeschreibung und müssen bei jeder Interpretation berücksichtigt werden:

- Die jeweils **jüngsten sieben Tage fehlen**.
- Angaben können vorläufig sein, auf Meldungen der berichtenden Personen beruhen und noch nicht verifiziert sein.
- Deliktklassifikationen können sich durch spätere Ermittlungen ändern.
- Mechanische oder menschliche Fehler sind möglich; CPD garantiert weder Vollständigkeit noch Genauigkeit oder zeitlich korrekte Reihenfolge.
- CPD weist ausdrücklich darauf hin, dass die Daten nicht für Vergleiche über die Zeit verwendet werden sollen. Zeitreihen müssen deshalb besonders vorsichtig interpretiert und diese Einschränkung muss im Bericht genannt werden.
- Adressen werden zum Schutz von Opfern nur auf Blockebene angezeigt.
- Kartenpositionen sind approximativ; aus ihnen dürfen keine konkreten Adressen rekonstruiert werden.
- Der Datensatz kann nachträglich aktualisiert werden. Ein wiederholter Abruf desselben Zeitraums kann deshalb veränderte Werte liefern.

Diese Quelle bildet **gemeldete und im System erfasste Straftaten** ab. Sie misst daher nicht automatisch die gesamte tatsächlich begangene Kriminalität.

## 8. Entscheidungen für die einfache Chicago-Ingestion

1. Wir lesen die Quelle über die offizielle SODA API.
2. Wir laden nicht sofort den vollständigen Datensatz seit 2001, sondern beginnen mit einem klar abgegrenzten Zeitraum.
3. Wir beschränken die Abfrage auf benötigte Spalten und einen klaren Zeitraum, übernehmen aber alle Deliktkategorien.
4. Wir verwenden `id` als technische Quellkennung und prüfen `case_number` zusätzlich auf Mehrfachvorkommen.
5. Wir speichern niemals den App Token im Code, Notebook oder Git.
6. Wir dokumentieren Abrufzeitpunkt, Filter und verwendeten API-Endpunkt.
7. Homicide bleibt enthalten, wird aber explizit mit Opfer-Granularität gekennzeichnet und in Auswertungen separat behandelt.

## 9. Offizielle Quellen

- [Chicago Data Portal: Crimes – 2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data)
- [Offizielle Metadaten des Datensatzes](https://data.cityofchicago.org/api/views/ijzp-q8t2)
- [Socrata API Foundry für `ijzp-q8t2`](https://dev.socrata.com/foundry/data.cityofchicago.org/ijzp-q8t2)
- [SODA Consumer API – Einstieg](https://dev.socrata.com/consumers/getting-started)
- [SODA-API-Endpunkte](https://dev.socrata.com/docs/endpoints)
- [SoQL-Abfragen mit SODA 3](https://dev.socrata.com/docs/queries/)
- [Socrata Application Tokens](https://dev.socrata.com/docs/app-tokens.html)
- [Chicago Police Department IUCR Codes](https://data.cityofchicago.org/Public-Safety/Chicago-Police-Department-Illinois-Uniform-Crime-R/c7ck-438e)

Die fachlichen Aussagen zur Abdeckung, Aktualisierung, Granularität, Herkunft aus CLEAR und zu den Einschränkungen basieren auf der offiziellen Datensatzbeschreibung. Technische Aussagen zur API basieren auf der offiziellen Socrata-Dokumentation.